In [1]:
# === 샘플 문서 준비 ===
# RAG 교재에서 사용한 sample_textbook.md의 내용 일부입니다.
document = """Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고,
사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."""

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# === 메시지 생성 ===
# SystemMessage: 모델의 행동 규칙을 지정합니다.
sys_msg = SystemMessage(content="다음 문서를 참고하여 질문에 답하세요. 문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답하세요.")

# HumanMessage: 문서와 질문을 함께 전달합니다.
human_msg = HumanMessage(content=f"[document]\n{document}\n\n[질문]\n어댑터즈에서 사용하는 교수법은 무엇인가요?")

# === 메시지 속성 확인 ===
print(f"type: {sys_msg.type}")
print(f"content 미리보기: {sys_msg.content[:40]}...")
print()
print(f"type: {human_msg.type}")
print(f"content 미리보기: {human_msg.content[:40]}...")

type: system
content 미리보기: 다음 문서를 참고하여 질문에 답하세요. 문서에 없는 내용은 '제공된 문서...

type: human
content 미리보기: [document]
Adapterz(어댑터즈)는 스타트업코드에서 운영하는...


In [4]:
# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# === 메시지 리스트로 호출 ===
messages = [sys_msg, human_msg]
response = model.invoke(messages)

print(f"응답: {response.content}")
print(f"응답 타입: {response.type}")

응답: 어댑터즈에서 사용하는 교수법은 5단 분석법입니다.
응답 타입: ai


In [5]:
# === 멀티턴 대화 히스토리 ===
# 이전 대화를 AIMessage로 포함하면 모델이 맥락을 이해합니다.
messages_with_history = [
    SystemMessage(content="다음 문서를 참고하여 질문에 답하세요. 문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답하세요."),
    HumanMessage(content=f"[document]\n{document}\n\n[질문]\n어댑터즈에서 사용하는 교수법은 무엇인가요?"),
    AIMessage(content="어댑터즈에서 사용하는 교수법은 5단 분석법입니다."),
    HumanMessage(content="그 교수법의 단계를 자세히 설명해 주세요.")
]

response = model.invoke(messages_with_history)
print(response.content)

5단 분석법은 기술 용어를 다음 단계로 나누어 체계적으로 설명하는 방식입니다:
*   기술 용어를 일반 명사와 고유 명사로 나누어 설명합니다.
*   사용 이유를 설명합니다.
*   사용 방법을 설명합니다.
*   다른 기술과의 비교를 통해 설명합니다.


In [6]:
# === 대화 히스토리에서 최근 N턴만 유지 ===
all_messages = [
    SystemMessage(content="다음 문서를 참고하여 질문에 답하세요."),
    HumanMessage(content=f"[document]\n{document}\n\n[질문]\n어댑터즈는 무엇인가요?"),
    AIMessage(content="어댑터즈는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다."),
    HumanMessage(content="어떤 분야의 교재를 제공하나요?"),
    AIMessage(content="인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."),
    HumanMessage(content="교수법 이름이 뭔가요?"),
    AIMessage(content="5단 분석법입니다."),
    HumanMessage(content="그 교수법의 단계를 설명해 주세요."),
]

# SystemMessage만 따로 뽑습니다.
# system 지시는 대화 전체에 공통으로 적용되는 규칙이므로, 트리밍할 때도 보통 유지합니다.
system = [m for m in all_messages if m.type == "system"]

# SystemMessage를 제외한 나머지 메시지들만 따로 뽑습니다.
# 여기에는 HumanMessage와 AIMessage가 순서대로 들어 있습니다.
non_system = [m for m in all_messages if m.type != "system"]

# 최근 4개 메시지만 남깁니다.
# 한 턴은 보통 HumanMessage 1개 + AIMessage 1개이므로, 최근 2턴은 메시지 4개를 슬라이싱하면 됩니다.
recent = non_system[-4:]

# 항상 유지할 SystemMessage와 최근 대화만 다시 합칩니다.
# 이렇게 하면 시스템 규칙은 유지하면서, 오래된 대화는 제거할 수 있습니다.
trimmed = system + recent

print(f"전체 메시지 수: {len(all_messages)}")
print(f"트리밍 후 메시지 수: {len(trimmed)}")
for m in trimmed:
    print(f"  [{m.type}] {m.content[:30]}")

전체 메시지 수: 8
트리밍 후 메시지 수: 5
  [system] 다음 문서를 참고하여 질문에 답하세요.
  [ai] 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재
  [human] 교수법 이름이 뭔가요?
  [ai] 5단 분석법입니다.
  [human] 그 교수법의 단계를 설명해 주세요.
